# v04 · OrbitCache: what the code does and what its docstring describes

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
vs = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/infinite-data-vector-shaders.py", "vs")
HW = vs.HyperWebster()
for t in ["listen", "silent", "enlist", "listen", "tinsel", "listen"]:
    HW.index_text(t)
print(HW.cache_stats())

{'total_lookups': 6, 'cache_hits': 2, 'cache_misses': 4, 'hit_rate': '33.3%', 'unique_strings': 4, 'multi_member_orbits': 1}


**What it does:** an exact-string cache plus a bookkeeping list of strings that are anagrams (same sorted letters). `listen`, `silent`, `enlist`, `tinsel` are one orbit under permutation of positions; each gets its own address, each computed by the full bijection. The module docstring describes reaching other orbit members 'by a cheap integer offset'; that offset is not implemented.

In [3]:
B = vs._Bijection()
addrs = {t: B.str_to_int(t) for t in ["listen", "silent", "enlist", "tinsel"]}
print(addrs)
print("hit rate above counts only exact repeats:", HW.cache_stats()["hit_rate"])

{'listen': 311118541415, 'silent': 251013532199, 'enlist': 150001668071, 'tinsel': 165148453895}
hit rate above counts only exact repeats: 33.3%


## The offset that *does* exist: a transposition
Swapping the characters at positions i < j changes the address by a closed form: `(d_j − d_i)·(N^(L−1−i) − N^(L−1−j))`, with `d = index + 1`. Checked here on a 500-character string.

In [4]:
import random
rnd = random.Random(7)
t = "".join(rnd.choice(B.char_list) for _ in range(500)); L = len(t); N = B.N
i, j = 17, 333
u = list(t); u[i], u[j] = u[j], u[i]; u = "".join(u)
d = lambda c: B._char_index[c] + 1
offset = (d(t[j]) - d(t[i])) * (N**(L-1-i) - N**(L-1-j))
assert B.str_to_int(u) == B.str_to_int(t) + offset
print("transposition offset identity holds; |offset| bits:", abs(offset).bit_length(), "| address bits:", B.str_to_int(t).bit_length())

transposition offset identity holds; |offset| bits: 3188 | address bits: 3299
